**Install the required libraries**

In [ ]:
# Install libraries to use matminer.
!pip install pyyaml -q
!pip install six -q
!pip install matminer[citrine] -q
!pip install citrination-client -q
!pip install --upgrade pandas==2.2.2 -q
!pip install pymatgen -q
!pip install --upgrade matplotlib==3.8.0 -q

# **1. Train and Test ML algorithm: Boston house price case**

**Loading libraries we want to use in this notebook**

(you don't need to load all of libraries at first and you can load it anytime and anywhere)

In [ ]:
import pandas as pd # a software library written for the Python programming language for data manipulation and analysis. In particular, it offers data structures and operations for manipulating numerical tables and time series
import seaborn as sns # Python data visualization library based on matplotlib. It provides a high-level interface for drawing attractive and informative statistical graphics
import numpy as np # a software library for the Python programming language, adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.
import matplotlib.pyplot as plt # a plotting library for the Python programming language

# sklearn is a machine learning software library for the Python programming language. It features various classification, regression and clustering algorithms including support vector machines, random forests, gradient boosting,
# k-means and DBSCAN, and is designed to interoperate with the Python numerical and scientific libraries NumPy and SciPy.
from sklearn.datasets import fetch_california_housing
from sklearn import linear_model, metrics, model_selection
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression

Loading the boston house-prices dataset from the sklearn repository

In [ ]:
# Load the California housing dataset
california = fetch_california_housing()

In [ ]:
# Print the key values of data
print(california.keys())
# Print the total row and column length of the dataset
print(california.data.shape)
# Print feature names
print(california.feature_names)

Check the description of dataset

In [ ]:
# Check the description of dataset
print(california.DESCR)

In [ ]:
df = pd.DataFrame(california.data)
df.head()

In [ ]:
# Enter feature as the column name
df.columns = california.feature_names
df.head()

In [ ]:
# Put the targetP('Price') in the dataset
df['Price'] = california.target
df.head()

**Feature selection**

Sometimes, the input variables may not be directly related to the interested target. Hence, a feature selection step is necessary. There are many different methods for selecting features. Here we will go over a simple implementation in scikit-learn.

In [ ]:
# Function that returns feature names
def identify_columns(x_new, nrows=10):
    columns = x_data.columns
    xvalues = x_data.values
    dist = np.linalg.norm(xvalues[:nrows, :, None] - x_new[:nrows, None, :], axis=0)
    return columns[np.argmin(dist, axis=0)].values

In [ ]:
# # x_data contains the features('california.feature_names')
x_data = df[california.feature_names]

# y_data contains the target variable('Price')
y_data = df['Price']

In [ ]:
# Select the top 5 features by using 'SelectKBest'
sel = SelectKBest(f_regression, k=5)
x_new = sel.fit_transform(x_data, y_data)
print(f"Selected features {identify_columns(x_new)}")

**1. Train ML algorithm**

In [ ]:
# Train about the average number of rooms and house prices by using 'LinearRegression'
linear_regression = linear_model.LinearRegression()
linear_regression.fit(X=pd.DataFrame(df['AveRooms']), y=df['Price'])
prediction = linear_regression.predict(X=pd.DataFrame(df['AveRooms']))

# Get the slope and intercept from the trained linear regression model
a = linear_regression.coef_
b = linear_regression.intercept_
df.plot(kind='scatter', x ="AveRooms", y="Price", figsize=(5, 5), color='black', xlim=(0,150), ylim=(0,5))
plt.tight_layout()
plt.plot(df['AveRooms'], prediction, color='b')
plt.text(20, 4, "Price = {:.3f} * AveRooms + {:.3f}".format(float(a), float(b)) , fontsize=15, color = "r")

**2. Train & Test ML algorithm**



In [ ]:
# Classify into training dataset and testing dataset using only "AveRooms" feature
x_data = df[['AveRooms']]
y_data = df['Price']

# Split the data into training and test sets('train_test_split')
x_train, x_test, y_train, y_test = model_selection.train_test_split(x_data, y_data, test_size=0.1)
print("Size of training set: {}, test set: {}".format(len(x_train), len(x_test)))

In [ ]:
# Train Linear regression model('fit')
estimator = linear_model.LinearRegression()
estimator.fit(X=x_train, y=y_train)

# Test with prediction by using R2 score('metrics.r2_score')
y_predict_test = estimator.predict(x_test)
score = metrics.r2_score(y_test, y_predict_test)

print('score: {:.2f}'.format(float(score)))
print('Mean Squared Error: {:.2f}'.format(metrics.mean_squared_error(y_predict_test, y_test)))
print('RMSE (* 1000$): {:.2f}'.format(metrics.mean_squared_error(y_predict_test, y_test)**.5))

# Print prediction values about test set
plt.figure(figsize=(4,4))
plt.scatter(y_test, y_predict_test, color='k')
plt.plot([0,10],[0,10])
plt.ylim(0, 10)
plt.ylim(0, 10)

In [ ]:
# Train with all features('california.feature_names')
x_data = df[california.feature_names]
y_data = df['Price']
x_train, x_test, y_train, y_test = model_selection.train_test_split(x_data, y_data, test_size=0.1)

estimator = linear_model.LinearRegression()
estimator.fit(x_train, y_train)

#y_predict_train = estimator.predict(x_train)
#score = metrics.r2_score(y_train, y_predict_train)
#print(score) #1.0

y_predict_test = estimator.predict(x_test)
score = metrics.r2_score(y_test, y_predict_test)

print('score: {:.2f}'.format(float(score)))
print('Mean Squared Error: {:.2f}'.format(metrics.mean_squared_error(y_predict_test, y_test)))
print('RMSE (* 1000$): {:.2f}'.format(metrics.mean_squared_error(y_predict_test, y_test)**.5))

plt.figure(figsize=(4,4))
plt.scatter(y_test, y_predict_test, color='k')
plt.plot([0,10],[0,10])
plt.ylim(0, 10)
plt.ylim(0, 10)

Benchmark the optimial number of features

In [ ]:
# Searching the optimal number of features by predicting R2 score while increasing the number of features
all = []
for i in range(1,len(california.feature_names)+1):
  sum_score = 0
  sum_mse = 0
  sum_rmse = 0
  x_data = df[california.feature_names]
  y_data = df['Price']
  sel = SelectKBest(f_regression, k=i)
  x_new = sel.fit_transform(x_data, y_data)
  selected = x_data.columns[sel.get_support()]
  print(f"Selected {i} features {selected}")
  x_data = df[selected]
  y_data = df['Price']
  # To consider changes of R2 score depending on random train set data due to the small number of data,
  # evaluate with the average of R2 score from 1000 iterations of training about the each number of features
  for j in range(1001):
    x_train, x_test, y_train, y_test = model_selection.train_test_split(x_data, y_data, test_size=0.1)
    estimator = linear_model.LinearRegression()
    estimator.fit(x_train, y_train)
    y_predict_test = estimator.predict(x_test)
    sum_score += float(metrics.r2_score(y_test, y_predict_test))
    sum_mse += float(metrics.mean_squared_error(y_predict_test, y_test))
    sum_rmse += float(metrics.mean_squared_error(y_predict_test, y_test)**.5)
 # print('average of score: {:.2f}'.format(sum_score/10))
 # print('average of Mean Squared Error: {:.2f}'.format(sum_mse/10))
 # print('average of RMSE (* 1000$): {:.2f}'.format(sum_rmse/10))
  all.append(sum_score/1000)
plt.plot([i for i in range(1, len(all)+1)], all)

**Benchmark the score (R2) with other various regression ML algorithm (LASSO, Ridge, Random Forest Tree)**

R2 score—varies between 0 and 100%. It is closely related to the mean square error, but not the same. R2 is the proportion of the variance in the dependent variable that is predictable from the independent variable(s). Another definition is “(total variance explained by model) / total variance.” So if it is 100%, the two variables are perfectly correlated, i.e., with no variance at all. A low value would show a low level of correlation, meaning a regression model that is not valid , but not in all cases.

In [ ]:
all = []
for i in range(1,len(california.feature_names)+1):
  sum_score = 0
  sum_mse = 0
  sum_rmse = 0
  x_data = df[california.feature_names]
  y_data = df['Price']
  sel = SelectKBest(f_regression, k=i)
  x_new = sel.fit_transform(x_data, y_data)
  selected = x_data.columns[sel.get_support()]
  print(f"Selected {i} features {selected}")
  x_data = df[selected]
  y_data = df['Price']
  for j in range(1001):
    x_train, x_test, y_train, y_test = model_selection.train_test_split(x_data, y_data, test_size=0.1)

    # For 'Lasso' regression
    estimator = linear_model.Lasso(alpha=0.35, max_iter=1000, tol=0.0001)
    # estimator = linear_model.Ridge(alpha=1)  # For Ridge regression
    # estimator = RandomForestRegressor()  # For Random forest regressor

    estimator.fit(x_train, y_train)
    y_predict_test = estimator.predict(x_test)
    sum_score += float(metrics.r2_score(y_test, y_predict_test))
    sum_mse += float(metrics.mean_squared_error(y_predict_test, y_test))
    sum_rmse += float(metrics.mean_squared_error(y_predict_test, y_test)**.5)
  print('average of score: {:.2f}'.format(sum_score/10))
  print('average of Mean Squared Error: {:.2f}'.format(sum_mse/10))
  print('average of RMSE (* 1000$): {:.2f}'.format(sum_rmse/10))
  all.append(sum_score/1000)
plt.plot([i for i in range(1, len(all)+1)], all)